# 05 - XGBoost (Gradient Boosting)

**Owner:** Member E

**Inputs:** preprocessed splits produced by `00_eda_and_preprocessing.ipynb`
(`artifacts/splits/splits.npz`). Do not re-split or re-scale here.

**Outputs:** `artifacts/results/xgboost.json`,
`artifacts/models/xgboost.joblib`,
plus ROC / PR / confusion-matrix PNGs in `figures/`.

**Why this model**

XGBoost is our **boosted-tree** model — the usual winner on tabular classification tasks of this size. Boosting fits trees sequentially to the residuals of the previous, capturing fine feature interactions. XGBoost has its own native imbalance knob, `scale_pos_weight`, which we use as Variant A. Variant B uses SMOTE the same way as the other notebooks for an apples-to-apples comparison. We use **RandomizedSearchCV(30 iters)** because the full grid (3*3*3*2*2 = 108 combos * 5 folds = 540 fits) is wasteful.

The structure below is identical across the five model notebooks:
1. Load shared splits.
2. **Variant A** — model with built-in imbalance handling (`class_weight`, `scale_pos_weight`, or distance weighting).
3. **Variant B** — SMOTE oversampling on training folds only (via `imblearn.pipeline.Pipeline`).
4. Pick the variant with the higher CV F1, refit, evaluate on the held-out test set.
5. Save the result JSON + the fitted model.


## 1. Setup & load shared splits

In [ ]:
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocess import load_splits
from src.tuning import grid_search, random_search, CV
from src.evaluation import (
    evaluate, save_results, print_metric_table,
    plot_confusion, plot_roc, plot_pr, _scores,
)

SPLITS_DIR  = PROJECT_ROOT / "artifacts" / "splits"
MODELS_DIR  = PROJECT_ROOT / "artifacts" / "models"
RESULTS_DIR = PROJECT_ROOT / "artifacts" / "results"
FIG_DIR     = PROJECT_ROOT / "figures"
for d in (MODELS_DIR, RESULTS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

data = load_splits(SPLITS_DIR)
X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]
feature_names   = data["feature_names"]

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"train fraud rate: {y_train.mean():.4f}, test fraud rate: {y_test.mean():.4f}")


## 2. ### Variant A: `scale_pos_weight` = neg/pos ratio

This is the boosting equivalent of class weighting — it scales the gradient contribution of positive samples up by the imbalance ratio.

In [ ]:
from xgboost import XGBClassifier

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight = {scale_pos_weight:.3f} (neg/pos ratio)")

estimator_cw = XGBClassifier(
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss",
    tree_method="hist",
    scale_pos_weight=scale_pos_weight,
)

dist_cw = {
    "n_estimators":     [200, 500, 1000],
    "learning_rate":    [0.05, 0.1, 0.2],
    "max_depth":        [3, 6, 10],
    "subsample":        [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0],
}

search_cw = random_search(estimator_cw, dist_cw, n_iter=30)
search_cw.fit(X_train, y_train)
print(f"Variant A best CV F1: {search_cw.best_score_:.4f}  params: {search_cw.best_params_}")


## 3. ### Variant B: SMOTE + XGBoost (no `scale_pos_weight`)

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

pipe_smote = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("clf",   XGBClassifier(
        random_state=42, n_jobs=-1, eval_metric="logloss", tree_method="hist",
    )),
])

dist_smote = {
    "clf__n_estimators":     [200, 500, 1000],
    "clf__learning_rate":    [0.05, 0.1, 0.2],
    "clf__max_depth":        [3, 6, 10],
    "clf__subsample":        [0.7, 1.0],
    "clf__colsample_bytree": [0.7, 1.0],
}

search_smote = random_search(pipe_smote, dist_smote, n_iter=30)
search_smote.fit(X_train, y_train)
print(f"Variant B best CV F1: {search_smote.best_score_:.4f}  params: {search_smote.best_params_}")


## 4. Pick the better variant, refit & evaluate on test

In [ ]:
# Pick the variant with the better CV F1 score
variants = {
    "class_weight": search_cw,
    "smote":        search_smote,
}
best_variant = max(variants, key=lambda k: variants[k].best_score_)
best_search  = variants[best_variant]
best_model   = best_search.best_estimator_

print(f"\nWinning variant: {best_variant}")
print(f"  CV F1 (winning):       {best_search.best_score_:.4f}")
print(f"  CV F1 (other variant): {variants['smote' if best_variant=='class_weight' else 'class_weight'].best_score_:.4f}")
print(f"  Best params:           {best_search.best_params_}")

# Final evaluation on the held-out test set
results = evaluate(
    best_model,
    X_train, y_train, X_test, y_test,
    model_name="xgboost",
    best_params=best_search.best_params_,
    imbalance_strategy=best_variant,
)
print_metric_table(results)

save_results(results, RESULTS_DIR / "xgboost.json")
joblib.dump(best_model, MODELS_DIR / "xgboost.joblib")
print(f"\nSaved -> artifacts/results/xgboost.json + artifacts/models/xgboost.joblib")


## 5. Diagnostic plots

In [ ]:
# Diagnostic plots
y_pred = best_model.predict(X_test)
y_score = _scores(best_model, X_test)

fig_cm = plot_confusion(y_test, y_pred, title=f"{results['model_name']} - confusion matrix")
fig_cm.savefig(FIG_DIR / "xgboost_confusion.png", bbox_inches="tight")
plt.show()

if y_score is not None:
    fig_roc = plot_roc(y_test, y_score, label="xgboost")
    fig_roc.savefig(FIG_DIR / "xgboost_roc.png", bbox_inches="tight")
    plt.show()

    fig_pr = plot_pr(y_test, y_score, label="xgboost")
    fig_pr.savefig(FIG_DIR / "xgboost_pr.png", bbox_inches="tight")
    plt.show()


## 6. Hand-off to the comparison notebook

`artifacts/results/xgboost.json` now contains:

```
model_name, imbalance_strategy, best_params,
train: {accuracy, balanced_accuracy, precision, recall, f1, roc_auc, pr_auc},
test:  {accuracy, balanced_accuracy, precision, recall, f1, roc_auc, pr_auc},
confusion_matrix_test
```

`06_comparison_and_interpretation.ipynb` will read this file (alongside the
other four) to build the comparison table and overlay the ROC / PR curves.